# [실습] LangChain으로 첫 체인 만들기

## 라이브러리 설치  

랭체인 관련 라이브러리를 설치합니다.  


In [8]:
# LangChain + LLM.......$@%^#*
!pip install langchain-huggingface langchain langchain-openai langchain-community langchain-ollama langchain-core langchain-google-genai #langchain-google-generativeai
!pip install dotenv langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 환경 변수 설정하기


In [1]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(".env", override=True)

True

모델 Provide의 클래스를 불러온 뒤, LLM을 연결하면 됩니다!

## Ollama를 통해 LLM 불러오기


https://ollama.com/


올라마는 가장 간단하게 오픈 모델을 구동할 수 있는 방식입니다.   
추론 최적화/병렬 처리 등의 기능은 뛰어나지 않지만  
CPU에서도 구동이 가능하며, UI도 지원합니다.

### Ollama 설치하고 서빙하기    
터미널을 이용해 아래의 커맨드를 입력합니다.

- 설치

```curl -fsSL https://ollama.com/install.sh | sh```

- Context Window 늘리기 (기본 Context 4096)

```export OLLAMA_CONTEXT_LENGTH=16384```

- 서빙 (& 으로 백그라운드 실행)

``` ollama serve &```



In [ ]:
#!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [2]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (18.7 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently 

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!ollama --version
!export OLLAMA_CONTEXT_LENGTH=16384
#!ollama serve &

올라마는 기본적으로 11434 포트에서 실행됩니다.   
Pull 을 통해 허깅페이스 모델의 주소나 Ollama 주소를 입력하면 모델을 불러와 저장합니다.

불러올 수 있는 모델의 목록은 https://ollama.com/search 에서 확인할 수 있습니다.   

기본적으로 양자화를 거친 상태로 불러오며, 16비트 Full Precision을 비롯한 다양한 버전도 확인할 수 있습니다.

터미널에서 아래 커맨드를 입력해 GPT-OSS 20B 모델을 불러옵니다.    
```
ollama pull gpt-oss:20b
```

In [ ]:
#!ollama pull gpt-oss:20b

Error: could not connect to ollama server, run 'ollama serve' to start it


In [5]:
import subprocess
import time

ollama_process = subprocess.Popen(["ollama", "serve"],
                                  stdout=subprocess.PIPE,
                                  stderr=subprocess.PIPE
                                  )


time.sleep(10)
print("Ollama server started")

Ollama server started


In [6]:
!ollama pull gpt-oss:20b

불러온 모델은 파일 시스템에 저장되며,   
Ollama 모듈을 호출할 때 해당 모델이 GPU에 로드되는 구조입니다.

In [9]:
#import sys
#!{sys.executable} -m pip install langchain-ollama
from langchain_ollama import ChatOllama
llm = ChatOllama(model='gpt-oss:20b',
                 temperature=0.1, reasoning=True)

test = '안녕!!!!!'
llm.invoke(test)

AIMessage(content='안녕!!! 😊 오늘은 어떤 도움을 드릴까요? 궁금한 점이나 필요한 정보가 있으면 언제든 말씀해 주세요!', additional_kwargs={'reasoning_content': 'The user says "안녕!!!!!" which is Korean greeting meaning "Hello!" with excitement. The assistant should respond in Korean presumably. It\'s a friendly greeting. We can reply with something like "안녕하세요! 어떻게 도와드릴까요?" or similar. Let\'s keep it friendly and ask how to help.'}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-06-29T00:48:56.345965247Z', 'done': True, 'done_reason': 'stop', 'total_duration': 104639032335, 'load_duration': 60517329604, 'prompt_eval_count': 70, 'prompt_eval_duration': 35631540000, 'eval_count': 101, 'eval_duration': 8475433000, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019f10d8-3155-7de3-b3bb-b70635750161-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 101, 'total_tokens': 171})

In [ ]:
test = 'ㅠㅠ'
for s in llm.stream(test):
    print(s.content, end='')

## Embedding


RAG의 구조는 검색을 수행하기 위한 임베딩 모델이 필요합니다.   
임베딩 모델은 텍스트를 벡터로 변환하며,    
이후 결과를 벡터 DB에 저장해 검색할 수 있습니다.

OpenAI의 `text-embedding-3-large` 는 빠른 속도로 연산이 가능하나, 비용이 발생하며 온라인 모델입니다.   
이에 따라, 폐쇄망/온프레미스 환경에서는 공개 임베딩 모델을 사용하여 구현해야 합니다.

In [10]:
from langchain_openai import OpenAIEmbeddings
openai_embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

허깅페이스에 게시된 공개 모델을 불러옵니다.   
오픈 임베딩 모델에서 중요한 파라미터는 다음과 같습니다.

- 파라미터 수 : 큰 임베딩 모델의 크기는 LLM에 육박합니다. GPU를 고려하여 선택합니다.
- Max Tokens: 임베딩 모델의 최대 토큰보다 큰 데이터를 입력하면, 앞부분만을 이용해 계산하게 되므로 적절한 검색이 되지 않을 수 있습니다.
- 임베딩 차원: 큰 차원의 벡터를 생성하는 임베딩 모델은 검색 속도가 감소합니다.

현재 한국어 데이터를 임베딩하기 위해 자주 사용하는 모델은 아래와 같습니다.

- BAAI/BGE-M3 (2GB, 8194 토큰 제한)   
BGE-M3 시리즈는 BAAI의 임베딩 모델로, 현재 가장 인기가 많은 모델입니다.

- nlpai-lab/KURE-v1 (2GB, 8194 토큰 제한)    
KURE 임베딩은 고려대학교 NLP 연구실에서 만든 모델로, BGE-M3를 한국어 텍스트로 파인 튜닝한 모델입니다.

- Qwen/Qwen-3-Embedding (0.6B, 4B, 8B, 32768 토큰 제한)    
알리바바 클라우드의 Qwen 모델을 개량하여 만든 모델입니다.    
가장 최신 모델로, BGE-M3와의 성능 비교가 치열합니다.

In [11]:
from sentence_transformers import SentenceTransformer
import torch

# HuggingFace 임베딩 주소 지정하기
# intfloat/multilingual-e5-small , baai/bge-m3, 등의 주소를 입력하여 지정
# GPU에 여유가 있다면 Qwen3 Embedding의 큰 사이즈 (4B, 8B)?

#model_name = 'Qwen/Qwen3-Embedding-0.6B'
model_name = 'Qwen/Qwen3-Embedding-4B'
#실제 주소: https://huggingface.co/BAAI/bge-m3

# CPU 설정으로 모델 불러오기
emb_model = SentenceTransformer(model_name,
            device='cuda', #'cpu',
            model_kwargs={'torch_dtype':torch.bfloat16})

# 로컬 폴더에 모델 저장하기
emb_model.save('./embedding')

# 모델 메모리에서 삭제
del emb_model
import gc
gc.collect()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

9617

파일 시스템에 저장한 오픈 모델은 HuggingFaceEmbeddings로 불러옵니다.

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

# 허깅페이스 포맷의 임베딩 모델 불러오기
open_embeddings = HuggingFaceEmbeddings(model_name= './embedding',
                                   model_kwargs={'device':'cuda', 'model_kwargs':{'torch_dtype':torch.bfloat16}})
                                                    # gpu 사용하기                               bfloat16

# GPU 로드된 것 확인
print('임베딩 모델 로드 완료!')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

임베딩 모델 로드 완료!


RAG를 하기 전, 비교를 위해 LLM에게 질문해 보겠습니다.

In [13]:
# Test
thinking = False
for s in llm.stream("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?"):
    if s.additional_kwargs.get('reasoning_content'):
        if not thinking:
            print('<THINK> ', end='')
            thinking=True
        print(s.additional_kwargs['reasoning_content'], end='')
    else:
        if thinking:
            print('</THINK>\n\n', end='')
            thinking=False
        print(s.content,end='')

<THINK> The user asks in Korean: "What is a domain-specific language model? What are some examples?" They want explanation of what a domain specialized language model is, and examples. Provide definition, characteristics, benefits, typical use cases, examples like BioBERT (biomedical), FinBERT (finance), Legal-BERT, CodeBERT for programming, SciBERT for scientific literature, etc. Also mention GPT-4 fine-tuned on specific domains. Provide explanation in Korean. Should be thorough but concise.</THINK>

## 도메인 특화 언어 모델(Domain‑Specific Language Model)이란?

### 1. 정의
**도메인 특화 언어 모델**은 일반적인 대규모 언어 모델(예: GPT‑3, BERT 등)을 특정 분야(도메인)의 텍스트에 맞춰 미세 조정(fine‑tuning)하거나, 그 도메인만을 대상으로 처음부터 학습시킨 모델입니다.  
- **특징**  
  - 해당 도메인의 전문 용어, 문체, 구조를 잘 이해합니다.  
  - 일반 모델보다 같은 작업(예: 질의응답, 요약, 분류)에서 더 높은 정확도와 신뢰성을 보입니다.  
  - 데이터가 제한적이거나 민감한 경우, 사전 학습된 가중치를 활용해 적은 데이터로도 좋은 성능을 낼 수 있습니다.

### 2. 왜 필요한가?
| 상황 | 일반 모델의 한계 | 도메인 특화 모델의 장점 |
|------|-----------------|------------------------|
| **전문 용어가 많음** (예: 의학, 법률

## 데이터 준비하기    
네이버 API를 통해, 검색어에 대한 뉴스 기사 링크를 가져오겠습니다.    


In [14]:
import requests
import os

def get_naver_news_links(query, num_links=100):
    """
    query와 num_links를 입력받아 네이버 검색 수행, 네이버 뉴스 URL의 기사만 수집
    """

    url = f"https://openapi.naver.com/v1/search/news.json?query={query}&display={num_links}&sort=sim"
    # 최대 100개의 결과를 표시

    NAVER_CLIENT_ID = 'g4yR7k_1CLoMOogWn3Q8'
    NAVER_CLIENT_SECRET = 'e9hdJO8V_2'

    client_id = NAVER_CLIENT_ID # os.getenv('NAVER_CLIENT_ID')
    client_secret = NAVER_CLIENT_SECRET # os.getenv('NAVER_CLIENT_SECRET')

    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret': client_secret
    }

    response = requests.get(url, headers=headers)
    result = response.json()

    # 특정 링크 형식만 필터링
    filtered_links = []
    for item in result['items']:
        link = item['link']
        if "n.news.naver.com/mnews/article/" in link:
            # 네이버 뉴스 스타일만 모으기
            filtered_links.append(link)

    # 결과 출력
    print(query, ':', len(filtered_links), 'Example:', filtered_links[0])
    # for link in filtered_links:
    #     print(link)

    return filtered_links

filtered_links = []
for topic in ['도메인 특화 언어모델', 'OpenAI', 'GPT', '구글', '가전제품', '넷플릭스']:
    filtered_links += get_naver_news_links(topic, 100)
print('Total Articles:', len(filtered_links))
print('Total Articles(Without Duplicate):',len(list(set(filtered_links))))
filtered_links = list(set(filtered_links))

도메인 특화 언어모델 : 46 Example: https://n.news.naver.com/mnews/article/138/0002232009?sid=105
OpenAI : 27 Example: https://n.news.naver.com/mnews/article/081/0003656388?sid=101
GPT : 78 Example: https://n.news.naver.com/mnews/article/009/0005699686?sid=105
구글 : 80 Example: https://n.news.naver.com/mnews/article/003/0014032937?sid=104
가전제품 : 57 Example: https://n.news.naver.com/mnews/article/022/0004138884?sid=101
넷플릭스 : 32 Example: https://n.news.naver.com/mnews/article/018/0006317488?sid=103
Total Articles: 320
Total Articles(Without Duplicate): 316


## LangChain Document Loaders

LangChain의 `document_loaders`는 다양한 형식의 파일을 불러올 수 있습니다.   
[https://python.langchain.com/docs/integrations/document_loaders/ ]    

Web URL로부터 페이지를 로드하는 기본 파서인 `WebBaseLoader`를 사용합니다.   

In [15]:
# Jupyter 분산 처리를 위한 설정
import nest_asyncio

nest_asyncio.apply()

In [16]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

async def get_news_documents(links):
    loader = WebBaseLoader(
        web_paths=links,
        bs_kwargs={'parse_only':bs4.SoupStrainer(class_=("newsct", "newsct-body"))},
                                # newsct, newsct-body만 추출 : 네이버 뉴스 포맷 HTML 요소

        requests_per_second = 10, # 1초에 10개 요청 보내기
        show_progress = True # 진행 상황 출력
    )
    # docs = loader.load() # 기본 코드
    # return docs

    docs = []

    async for doc in loader.alazy_load():
        # 순차적 로드 대신 비동기 처리
        docs.append(doc)
    return docs

docs = await get_news_documents(filtered_links)

/tmp/ipykernel_5458/1843925440.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
Fetching pages: 100%|##########| 316/316 [02:09<00:00,  2.44it/s]


In [17]:
from rich import print as rprint
rprint(docs[12])

Document(
    metadata={'source': 'https://n.news.naver.com/mnews/article/374/0000518676?sid=104'},
    page_content='\n\n\n\n\n\n\nSBS Biz\n\n\n구독\n\nSBS Biz 언론사 구독되었습니다. 메인 뉴스판에서  주요뉴스를  볼
수 있습니다.\n보러가기\n닫기\n\n\nSBS Biz 언론사 구독 해지되었습니다.\n닫기\n\n\n\n\n[글로벌 비즈 브리핑] 오픈AI, 
새 AI 모델 공개…美 정부 요청에 일부기관 제공 外\n\n\n\n\n\n\n\n\n입력\n2026.06.29. 오전 3:29\n\n\n수정\n2026.06.29.
오전 5:48\n\n\n\n기사원문\n 
\n\n\n\n\n\n\n\n\n\n\n추천\n반응\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n
0\n\n\n\n\n후속강추\n0\n\n\n \n\n\n\n\n댓글\n반응\n\n\n\n\n\n\n\n\n텍스트 음성 변환 서비스 
사용하기\n\n\n\n성별\n남성\n여성\n\n\n말하기 속도\n느림\n보통\n빠름\n\n이동 통신망을 이용하여 음성을 재생하면 
별도의 데이터 통화료가 부과될 수 있습니다.\n본문듣기 시작\n\n닫기\n\n\n \n\n글자 크기 
변경하기\n\n글자크기\n\n\n가1단계\n작게\n\n\n가2단계\n보통\n\n\n가3단계\n크게\n\n\n가4단계\n아주크게\n\n\n가5단계\n
최대크게\n\n\n닫기\n\n\n\n\nSNS 보내기\n\n\n\n인쇄하기\n\n\n\n\n\n\n\n\n\n\n\n\n[오픈AI 로고 일러스트 
(로이터=연합뉴스)][글로벌 비즈 브리핑] 한 눈에 보는 해외 경제 이슈▲메모리 대란에...애플 "中 칩 구해달라" 美 정부에 
로비▲오픈AI, 새 AI 모델 공개...美 정부 요청에 일부기관 제공▲"AI 붐, 장기 투자 실패로 끝날 위험:...섬뜩한 
경고▲한사람당 100억원 벌었다...日 키옥시아 직원들 \'잭팟\'▲스페이스X, 모바일 통신사업 진출 시동...차터와 
논의▲"비트코인, 소리없이 사라질 것"...닷컴버블 맞힌 그랜섬의 경고메모리 대란에...애플 "中 칩 구해달라" 美 정부에 
로비세계적인 메모리 반도체 품귀 탓에 생산 비용이 크게 늘어난 애플이 미국 정부의 블랙리스트에 오른 중국산 메모리 칩 
구매를 적극 검토하는 것으로 전해졌습니다.현지시간 26일 파이낸셜타임스(FT)는 애플이 미국 상무부를 비롯한 도널드 
트럼프 행정부 관계자들을 상대로 중국산 메모리 칩 구매를 승인해달라는 로비를 벌이고 있다고 보도했습니다.애플이 
관심을 보이는 중국의 반도체 업체는 창신메모리테크놀로지(CXMT)인 것으로 알려졌습니다.D램 제조사인 CXMT는 중국 
인민해방군과 연계됐다는 이유로 미 국방부가 ‘중국 군사기업(1260H)’이라는 일종의 블랙리스트에 올려놓은 업체입니다.이 
명단에 포함될 경우 직접적인 법적 제재를 받습니다. 하지만 거래 기업은 평판 리스크를 감수해야 합니다.애플이 워싱턴 
정가를 상대로 로비에 나선 것은 메모리 가격 급등으로 공급망 다변화를 추진해야 하는 상황이기 때문입니다.그러나 애플의
로비가 성공할지 여부는 미지수입니다. 특히 미국 의회의 반발이 변수로 꼽힙니다.하원 중국특별위원회의 존 
물레나(공화·미시간) 위원장은 “애플이 중국 군사 기업과 협력하는 것은 심각한 실수가 될 것”이라며 “중국 공산당이 핵심 
공급망을 장악하도록 돕는 결과를 초래할 수 있다”고 비판했습니다.마코 루비오 국무장관도 상원의원이던 지난 2022년 
애플이 중국의 YMTC 메모리 칩 채택을 검토했을 당시 “불장난을 하는 것”이라며 강하게 반대한 바 있습니다.낸드플래시 
메모리 전문 업체인 YMTC는 중국 군사기업 블랙리스트에 올라와 있는 업체입니다.오픈AI, 새 AI 모델 공개...美 정부 
요청에 일부기관 제공오픈AI가 차세대 인공지능(AI) 모델 \'GPT-5.6\' 제품을 출시하면서, 미국 정부의 요청으로 일부 
기관에만 제한적으로 우선 제공하기로 했습니다.오픈AI는 새 AI 모델 \'GPT-5.6\'을 \'솔\', \'테라\', \'루나\' 등 세 
가지 세부모델로 나눠 개발했다고 현지시간 26일 밝혔습니다.이 가운데 상위 모델인 \'솔\' 모델은 코딩, 생물학, 사이버 
보안 등 부문에서 역대 최고 성능과 자율적 에이전트 능력을 갖췄다고 오픈AI는 설명했습니다.오픈AI는 이 모델의 추론 
능력을 강화하기 위해 모델에 역대 가장 긴 시간을 부여하는 \'최대 추론 노력\' 옵션과 하위 에이전트를 활용해 복잡한 
작업을 좀 더 빠르게 수행하는 \'울트라 모드\'도 새로 도입했습니다.오픈AI는 터미널 코딩 능력을 재는 \'터미널-벤치 
2.1\' 성능지표(벤치마크)에서 GPT-5.6 솔 울트라가 91.9%, GPT-5.6 솔이 88.8%를 기록해 라이벌 앤트로픽의 클로드 
미토스5의 88%를 능가했다고도 강조했습니다.특히 오픈AI는 솔 모델이 현재까지 가장 견고한 안전 체계를 갖춰 고위험 
활동이나 해킹에 악용될 우려가 있는 민감한 사이버 요청, 오용 등에 대한 보호 조치를 강화했다고도 
강조했습니다.오픈AI는 그간 새 모델 발표 때마다 제공해 왔던 세부적인 성능지표는 공개하지 않았습니다.중위 세부 모델인
\'테라\'는 전작인 GPT-5.5와 유사한 성능을 보이면서도 가격은 절반으로 낮췄고, 하위 모델인 \'루나\'는 최저 비용으로 
준수한 성능을 제공한다고 밝혔습니다.오픈AI는 이들 모델을 우선 정부와 공유된 \'신뢰할 수 있는 파트너\'를 대상으로 선
제공한다면서 실제 일반 배포 시기는 몇 주 뒤가 될 것이라고 예고했습니다.다만 오픈AI는 "이러한 형태의 정부 승인 
절차가 장기적인 표준이 되어서는 안 된다고 생각한다"며 "해당 도구가 필요한 사용자·개발자·기업·사이버 보안 
전문가·글로벌 파트너들이 최고의 도구를 활용하지 못하게 막기 때문"이라고 덧붙였습니다.도널드 트럼프 대통령은 이달 초
AI 기업이 새 모델을 공개하기 최대 30일 전에 정부에 제출할 수 있도록 하는 내용의 행정명령에 서명했고, 미 상무부는 
앤트로픽의 첨단 모델 \'미토스5\'와 \'페이블5\'가 국가 안보 위험을 초래한다며 수출 통제 지침을 내놨습니다.한편 
오픈AI는 당초 올해로 예상됐던 상장 추진 일정을 늦추고 있는 것으로 전해졌습니다.이달 초 증권거래위원회(SEC)에 비공개
기업공개(IPO) 신청서를 제출한 오픈AI는 아직 잠재적인 공모 가격과 수요 등을 논의하기 위한 사전 회의도 열지 않았고, 
상장 일정에 대한 공식 계획도 제시하지 않았다고 미 경제방송 CNBC가 소식통을 인용해 이날 보도했습니다.앞서 
뉴욕타임스(NYT)는 오픈AI가 연내 추진하려 했던 상장 시기를 내년으로 미루는 쪽으로 기울고 있다고 전날 보도한 바 
있습니다."AI 붐, 장기 투자 실패로 끝날 위험:...섬뜩한 경고빅테크의 인공지능(AI) 투자 광풍이 장기적인 투자 붕괴로 
끝날 위험이 있다고 국제결제은행(BIS)이 경고했습니다.28일(현지시간) 파이낸셜타임스(FT)에 따르면 BIS는 이날 ‘연례 
경제 보고서’를 통해 기술 부문의 수익률이 기대에 못 미칠 가능성이 커질 경우 투자자들이 AI 기업에 대한 자금 공급을 
급격히 줄일 수 있다고 내다봤습니다. 5대 대형 하이퍼스케일러(초대형 클라우드 사업자)들은 2025년부터 2026년 말까지 
1조달러 이상을 투자할 것으로 예상됩니다.BIS는 현재의 AI 호황이 안고 있는 위험을 설명하며 “수익률에 대한 실망은 자금
조달의 갑작스러운 후퇴를 촉발할 수 있고, 자본지출 붐을 장기적인 투자 침체로 바꿀 수 있다”며 “이는 금융 여건에도 
연쇄적인 영향을 미칠 수 있다”고 분석했습니다.이는 AI 투자를 위한 주식·채권 발행 규모가 커지고, 이로 인해 글로벌 
시장의 변동성이 확대되는 데 대한 우려가 커지는 가운데 나왔습니다. 기술기업들은 AI 프로젝트 자금을 마련하기 위해 
신용시장으로 몰려들어 수천억달러를 조달했습니다.사상 최고 수준의 주가도 이들을 미국 주식시장으로 끌어들이고 
있습니다. 이달 초 스페이스X가 860억달러 규모

불필요한 내용을 전처리합니다.

In [18]:
import re

def preprocess(docs):
    noise_texts = [
        '''구독중 구독자 0 응원수 0 더보기''',
        '''쏠쏠정보 0 흥미진진 0 공감백배 0 분석탁월 0 후속강추 0''',
        '''댓글 본문 요약봇 본문 요약봇''',
        '''도움말 자동 추출 기술로 요약된 내용입니다. 요약 기술의 특성상 본문의 주요 내용이 제외될 수 있어, 전체 맥락을 이해하기 위해서는 기사 본문 전체보기를 권장합니다. 닫기''',
        '''텍스트 음성 변환 서비스 사용하기 성별 남성 여성 말하기 속도 느림 보통 빠름''',
        '''이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다. 본문듣기 시작''',
        '''닫기 글자 크기 변경하기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 SNS 보내기 인쇄하기''',
        'PICK 안내 언론사가 주요기사로선정한 기사입니다. 언론사별 바로가기 닫기',
        '응원 닫기',
        '구독 구독중 구독자 0 응원수 0 ',

    ]

    def clean_text(doc):
        text = doc.page_content
        # 탭과 개행문자를 공백으로 변환
        text = text.replace('\t', ' ').replace('\n', ' ')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()

        # 여러 구분자를 한번에 처리
        split_markers = [
            '구독 해지되었습니다.',
            '구독 메인에서 바로 보는 언론사 편집 뉴스 지금 바로 구독해보세요!'
        ]


        for marker in split_markers:
            parts = text.split(marker)
            if len(parts) > 1:
                if marker == '구독 해지되었습니다.':
                    text = parts[1]  # 뒷부분 사용
                else:
                    text = parts[0]  # 앞부분 사용


        # 노이즈 텍스트 제거
        for noise in noise_texts:
            text = text.replace(noise, '')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()

        doc.page_content = text

        return doc


    preprocessed_docs = []
    for doc in docs:

        # 텍스트 정제
        doc= clean_text(doc)
        preprocessed_docs.append(doc)

    return preprocessed_docs

preprocessed_docs = preprocess(docs)

In [19]:
rprint(preprocessed_docs[12])

Document(
    metadata={'source': 'https://n.news.naver.com/mnews/article/374/0000518676?sid=104'},
    page_content='닫기 [글로벌 비즈 브리핑] 오픈AI, 새 AI 모델 공개…美 정부 요청에 일부기관 제공 外 입력 
2026.06.29. 오전 3:29 수정 2026.06.29. 오전 5:48 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 
가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 [오픈AI 로고 
일러스트 (로이터=연합뉴스)][글로벌 비즈 브리핑] 한 눈에 보는 해외 경제 이슈▲메모리 대란에...애플 "中 칩 구해달라" 
美 정부에 로비▲오픈AI, 새 AI 모델 공개...美 정부 요청에 일부기관 제공▲"AI 붐, 장기 투자 실패로 끝날 위험:...섬뜩한 
경고▲한사람당 100억원 벌었다...日 키옥시아 직원들 \'잭팟\'▲스페이스X, 모바일 통신사업 진출 시동...차터와 
논의▲"비트코인, 소리없이 사라질 것"...닷컴버블 맞힌 그랜섬의 경고메모리 대란에...애플 "中 칩 구해달라" 美 정부에 
로비세계적인 메모리 반도체 품귀 탓에 생산 비용이 크게 늘어난 애플이 미국 정부의 블랙리스트에 오른 중국산 메모리 칩 
구매를 적극 검토하는 것으로 전해졌습니다.현지시간 26일 파이낸셜타임스(FT)는 애플이 미국 상무부를 비롯한 도널드 
트럼프 행정부 관계자들을 상대로 중국산 메모리 칩 구매를 승인해달라는 로비를 벌이고 있다고 보도했습니다.애플이 
관심을 보이는 중국의 반도체 업체는 창신메모리테크놀로지(CXMT)인 것으로 알려졌습니다.D램 제조사인 CXMT는 중국 
인민해방군과 연계됐다는 이유로 미 국방부가 ‘중국 군사기업(1260H)’이라는 일종의 블랙리스트에 올려놓은 업체입니다.이 
명단에 포함될 경우 직접적인 법적 제재를 받습니다. 하지만 거래 기업은 평판 리스크를 감수해야 합니다.애플이 워싱턴 
정가를 상대로 로비에 나선 것은 메모리 가격 급등으로 공급망 다변화를 추진해야 하는 상황이기 때문입니다.그러나 애플의
로비가 성공할지 여부는 미지수입니다. 특히 미국 의회의 반발이 변수로 꼽힙니다.하원 중국특별위원회의 존 
물레나(공화·미시간) 위원장은 “애플이 중국 군사 기업과 협력하는 것은 심각한 실수가 될 것”이라며 “중국 공산당이 핵심 
공급망을 장악하도록 돕는 결과를 초래할 수 있다”고 비판했습니다.마코 루비오 국무장관도 상원의원이던 지난 2022년 
애플이 중국의 YMTC 메모리 칩 채택을 검토했을 당시 “불장난을 하는 것”이라며 강하게 반대한 바 있습니다.낸드플래시 
메모리 전문 업체인 YMTC는 중국 군사기업 블랙리스트에 올라와 있는 업체입니다.오픈AI, 새 AI 모델 공개...美 정부 
요청에 일부기관 제공오픈AI가 차세대 인공지능(AI) 모델 \'GPT-5.6\' 제품을 출시하면서, 미국 정부의 요청으로 일부 
기관에만 제한적으로 우선 제공하기로 했습니다.오픈AI는 새 AI 모델 \'GPT-5.6\'을 \'솔\', \'테라\', \'루나\' 등 세 
가지 세부모델로 나눠 개발했다고 현지시간 26일 밝혔습니다.이 가운데 상위 모델인 \'솔\' 모델은 코딩, 생물학, 사이버 
보안 등 부문에서 역대 최고 성능과 자율적 에이전트 능력을 갖췄다고 오픈AI는 설명했습니다.오픈AI는 이 모델의 추론 
능력을 강화하기 위해 모델에 역대 가장 긴 시간을 부여하는 \'최대 추론 노력\' 옵션과 하위 에이전트를 활용해 복잡한 
작업을 좀 더 빠르게 수행하는 \'울트라 모드\'도 새로 도입했습니다.오픈AI는 터미널 코딩 능력을 재는 \'터미널-벤치 
2.1\' 성능지표(벤치마크)에서 GPT-5.6 솔 울트라가 91.9%, GPT-5.6 솔이 88.8%를 기록해 라이벌 앤트로픽의 클로드 
미토스5의 88%를 능가했다고도 강조했습니다.특히 오픈AI는 솔 모델이 현재까지 가장 견고한 안전 체계를 갖춰 고위험 
활동이나 해킹에 악용될 우려가 있는 민감한 사이버 요청, 오용 등에 대한 보호 조치를 강화했다고도 
강조했습니다.오픈AI는 그간 새 모델 발표 때마다 제공해 왔던 세부적인 성능지표는 공개하지 않았습니다.중위 세부 모델인
\'테라\'는 전작인 GPT-5.5와 유사한 성능을 보이면서도 가격은 절반으로 낮췄고, 하위 모델인 \'루나\'는 최저 비용으로 
준수한 성능을 제공한다고 밝혔습니다.오픈AI는 이들 모델을 우선 정부와 공유된 \'신뢰할 수 있는 파트너\'를 대상으로 선
제공한다면서 실제 일반 배포 시기는 몇 주 뒤가 될 것이라고 예고했습니다.다만 오픈AI는 "이러한 형태의 정부 승인 
절차가 장기적인 표준이 되어서는 안 된다고 생각한다"며 "해당 도구가 필요한 사용자·개발자·기업·사이버 보안 
전문가·글로벌 파트너들이 최고의 도구를 활용하지 못하게 막기 때문"이라고 덧붙였습니다.도널드 트럼프 대통령은 이달 초
AI 기업이 새 모델을 공개하기 최대 30일 전에 정부에 제출할 수 있도록 하는 내용의 행정명령에 서명했고, 미 상무부는 
앤트로픽의 첨단 모델 \'미토스5\'와 \'페이블5\'가 국가 안보 위험을 초래한다며 수출 통제 지침을 내놨습니다.한편 
오픈AI는 당초 올해로 예상됐던 상장 추진 일정을 늦추고 있는 것으로 전해졌습니다.이달 초 증권거래위원회(SEC)에 비공개
기업공개(IPO) 신청서를 제출한 오픈AI는 아직 잠재적인 공모 가격과 수요 등을 논의하기 위한 사전 회의도 열지 않았고, 
상장 일정에 대한 공식 계획도 제시하지 않았다고 미 경제방송 CNBC가 소식통을 인용해 이날 보도했습니다.앞서 
뉴욕타임스(NYT)는 오픈AI가 연내 추진하려 했던 상장 시기를 내년으로 미루는 쪽으로 기울고 있다고 전날 보도한 바 
있습니다."AI 붐, 장기 투자 실패로 끝날 위험:...섬뜩한 경고빅테크의 인공지능(AI) 투자 광풍이 장기적인 투자 붕괴로 
끝날 위험이 있다고 국제결제은행(BIS)이 경고했습니다.28일(현지시간) 파이낸셜타임스(FT)에 따르면 BIS는 이날 ‘연례 
경제 보고서’를 통해 기술 부문의 수익률이 기대에 못 미칠 가능성이 커질 경우 투자자들이 AI 기업에 대한 자금 공급을 
급격히 줄일 수 있다고 내다봤습니다. 5대 대형 하이퍼스케일러(초대형 클라우드 사업자)들은 2025년부터 2026년 말까지 
1조달러 이상을 투자할 것으로 예상됩니다.BIS는 현재의 AI 호황이 안고 있는 위험을 설명하며 “수익률에 대한 실망은 자금
조달의 갑작스러운 후퇴를 촉발할 수 있고, 자본지출 붐을 장기적인 투자 침체로 바꿀 수 있다”며 “이는 금융 여건에도 
연쇄적인 영향을 미칠 수 있다”고 분석했습니다.이는 AI 투자를 위한 주식·채권 발행 규모가 커지고, 이로 인해 글로벌 
시장의 변동성이 확대되는 데 대한 우려가 커지는 가운데 나왔습니다. 기술기업들은 AI 프로젝트 자금을 마련하기 위해 
신용시장으로 몰려들어 수천억달러를 조달했습니다.사상 최고 수준의 주가도 이들을 미국 주식시장으로 끌어들이고 
있습니다. 이달 초 스페이스X가 860억달러 규모의 초대형 기업공개(IPO)를 단행한 것은 AI 관련 주식에 대한 폭발적인 
수요를 보여주는 상징적 사례입니다.투자자들은 AI 투자가 충분한 수익을 내지 못할 경우 이 같은 채권 발행 러시가 역풍을
초래할 수 있다고 경고해왔습니다. 스페이스X IPO 이후 주식시장은 변동성이 커졌고, 연방준비제도(연준, Fed)의 금리 인상
전망에 무게가 실리고 있습니다. 일각에선 스페이스X가 IPO 직후 250억달러 규모의 채권 발행에 나선 것을 두고 “시장이 
버블 영역에 진입했다는 신호”라고 경고했습니다.BIS는 지금까지 AI에 대한 낙관론이 세계 성장에 중요한 순풍으로 
작용했다고 짚었습니다. 보고서는 AI가 기업들에 제공할 수 있는 효율성 향상을 감안하면, 향후 10년 동안 생산성을 상당히
끌어올릴 가능성이 있다고 인정했습니다.그럼에도 BIS는 과거 투자 붐 사례들이 “시사하는 바가 크다”고 지적했다. 예로는 
1830년대 운하 건설 확대, 1840년대 영국 철도 붐, 1990년대 말 닷컴 붐을 들었습니다.BIS는 “이들 사례는 

불러온 텍스트 데이터는 파일로 저장할 수 있습니다.

In [20]:
rprint(preprocessed_docs[12].metadata)

{'source': 'https://n.news.naver.com/mnews/article/374/0000518676?sid=104'}

In [21]:
!pip install jsonlines langchain langchain-core

In [22]:
# 불러온 document 저장하기

import jsonlines

def save_docs_to_jsonl(documents, file_path):
    with jsonlines.open(file_path, mode="w") as writer:
        for doc in documents:
            writer.write(doc.model_dump())

save_docs_to_jsonl(preprocessed_docs, "docs.jsonl")

# jsonl 파일 불러오기
#from langchain.schema import Document
from langchain_core.documents import Document

def load_docs_from_jsonl(file_path):
    documents = []
    with jsonlines.open(file_path, mode="r") as reader:
        for doc in reader:
            documents.append(Document(**doc))
    return documents

평가 데이터와의 연결을 위해, 전처리한 데이터 대신 실습 시트의 데이터를 사용하겠습니다.

In [23]:
preprocessed_docs = load_docs_from_jsonl("docs.jsonl") #("eval.jsonl")

In [24]:
preprocessed_docs[2]

Document(metadata={'source': 'https://n.news.naver.com/mnews/article/056/0012206804?sid=104'}, page_content='닫기 미국, AI 경쟁의 심장부…구글 데이터센터 첫 공개 입력 2026.06.26. 오후 12:44 수정 2026.06.26. 오후 12:52 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 철저한 보안 절차를 통과해 들어선 미국 앨라배마주의 구글 AI 데이터센터. 끝이 보이지 않는 서버마다 수천 개의 첨단 반도체가 24시간 쉬지 않고 돌아갑니다.[카린 리들리/구글 사이트 운영 매니저 : "G메일을 확인하거나 유튜브 영상을 보고 지도를 검색하는 모든 작업이 바로 이 서버에서 처리됩니다."] AI 수요가 폭증하면서 빅테크 기업들의 데이터센터 확장 경쟁도 갈수록 치열해지고 있습니다. 구글 역시 앨라배마의 240만㎡ 규모 데이터센터 단지를 확장하고 있습니다. 하지만 막대한 전력과 물 사용, 소음 문제를 둘러싼 우려도 커지고 있습니다. 최근 미국 여론조사에서는 응답자의 70% 이상이 자신의 지역에 AI 데이터센터가 들어서는데 반대한다고 답했습니다.[토머스 갬블/구글 데이터센터 관리자 : "부정적인 시선이 많습니다. 그래서 전력, 물과 자원을 최대한 효율적으로 활용하는 방안을 찾고 있습니다."] 구글은 자체 변전소를 구축해 전력 비용 부담 논란을 줄이겠다고 밝혔지만, AI 데이터센터를 둘러싼 논란은 계속되고 있습니다. ■ 제보하기▷ 전화 : 02-781-1234, 4444▷ 이메일 : kbs1234@kbs.co.kr▷ 카카오톡 : \'KBS제보\' 검색, 채널 추가▷ 유튜브, 다음에서도 KBS뉴스를 구독해주세요! Copyright ⓒ KBS. All rights reserved. 무단 전재, 재배포 및 이용(AI 학습 포함) 금지 이 기사는 언론사에서 세계

## Chunking: 청크 단위로 나누기   



전처리가 완료된 docs를 chunk 단위로 분리합니다.
`chunk_size`와 `chunk_overlap`을 이용해 청크의 구성 방식을 조절할 수 있습니다.  

Chunk Size * K(검색할 청크의 수) 의 결과가 Context의 길이가 됩니다.

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain import hub

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# 0~1000, 800~1800, 1600~2600, ...
# 엔터, 공백, 쉼표 등의 구분자(delimiter)를 기준으로 최대한 청킹
chunks = text_splitter.split_documents(preprocessed_docs)
print(len(chunks))

# Top 5 검색이라면? --> 5 * 1000 = 5000 글자 Context

1071


## Vector Database 만들기   

구성된 청크를 벡터 데이터베이스에 로드합니다.   

In [26]:
!pip install langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 142.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opente

In [27]:
from langchain_chroma import Chroma
import uuid

Chroma().delete_collection() # (메모리에 저장하는 경우) 기존 데이터 삭제

uuidstr = str(uuid.uuid4())[0:6]
# 랜덤 식별자

# DB 구성하기
db = Chroma(embedding_function=openai_embeddings,
            persist_directory=f"./chroma_OpenAI_{uuidstr}",
            # 파일 시스템에 저장 (생략시 메모리에 저장)

            collection_name='Web', # 식별 이름

            collection_metadata={'hnsw:space':'l2'},
            # l2 메트릭 설정(기본값, cosine, mmr 로 변경 가능)
            )

ImportError: cannot import name 'OTEL_PYTHON_SDK_INTERNAL_METRICS_ENABLED' from 'opentelemetry.sdk.environment_variables' (/usr/local/lib/python3.12/dist-packages/opentelemetry/sdk/environment_variables/__init__.py)

DB에 document를 추가합니다.    
OpenAI 임베딩은 30만 토큰 동시 처리 제한이 있어, 나눠서 전달합니다.

In [ ]:
from tqdm import tqdm

print(len(chunks))
# 300,000 토큰 제한

# 100개씩 추가
for i in tqdm(range(0, len(chunks), 100)):
    db.add_documents(chunks[i:min(i+100, len(chunks))])

993


100%|██████████| 10/10 [00:31<00:00,  3.13s/it]


db로부터 retriever를 구성합니다.

In [ ]:
# Top 5 Search(기본값은 4)
retriever = db.as_retriever(search_kwargs={'k':5})

In [ ]:
retriever.invoke("도메인 특화 언어 모델")

[Document(id='5bf27a03-09bc-470f-bb5f-b759d7a61303', metadata={'source': 'https://n.news.naver.com/mnews/article/047/0002520379?sid=102'}, page_content="도메인 지식은 100% 자사 서버에 축적하여 데이터 주권을 확보해야 합니다. 외국 모델을 일종의 '데이터 수집 및 워크플로우 검증용 프로브'로 활용하는 것입니다.무엇보다 본질적인 것은, 각 산업 단위에서 AX(AI Transformation)가 단순한 기술 도입을 넘어 기존 가치 사슬을 해체하고 재편하는 전략적 필연성임을 경영진이 통찰하는 것입니다. 이를 위해 단순한 API 호출을 넘어, 현장의 도메인 지식(Domain Knowledge)을 AI 에이전트의 워크플로우로 구조화하고, 예외 상황을 디버깅하며, 실증 기반으로 모델 성능을 지속 최적화하는 '기업 특화 AI 운영 역량(AIOps 및 도메인 파인튜닝 노하우)'을 조직 내부에 내재화해야 합니다.이 과정에서 확보된 운영 철학과 축적된 고품질 도메인 데이터, 자체 평가 파이프라인은 향후 독자 파운데이션 모델이 등장했을 때 가장 강력한 부스터가 될 것입니다.대형 모델의 경쟁이 무대 위의 데모에서 실제 생산 라인의 토큰 소비량으로 옮겨가고 있습니다. 모델은 언제든 교체할 수 있지만 구축된 에이전트 워크플로우와 내재화된 운영 노하우, 그리고 축적된 도메인 데이터는 영원한 자산입니다. 180조 토큰의 위력은 본격적인 AX 혁명의 신호탄이며, 우리도 더 이상 미루면 늦습니다. 이 시스템적 철학을 바탕으로 거대한 생산성 혁명에 동참해야 할 결정적 시점에 서 있습니다.덧붙이는 글 | 임선영 씨는 중국 칭화대 전산언어학 석사를 마친 중국경제전문가이며 <중국경제 미래지도>, <중국AI 미래지도>의 저자입니다. 이 글은 본인의 페북에도 올렸습니다. Copyright ⓒ 오마이뉴스. All rights reserved. 무단 전재 및 재배포 금지. 이 기사는 

## Prompting

RAG를 위한 간단한 프롬프트를 작성합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate([
    ("system", '''당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.
Question과 무관한 내용이 들어 있을 수 있으므로 신중히 판단하세요.'''),
('human','''
---
Context: {context}
---
Question: {question}''')])
prompt.pretty_print()

================================ System Message ================================

당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.
Question과 무관한 내용이 들어 있을 수 있으므로 신중히 판단하세요.

================================ Human Message =================================


---
Context: {context}
---
Question: {question}


## Chain

RAG를 수행하기 위한 Chain을 만듭니다.

RAG Chain은 프롬프트에 context와 question을 전달해야 합니다.    
체인의 입력은 Question만 들어가므로, Context를 동시에 prompt에 넣기 위해서는 아래의 구성이 필요합니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# retriever의 결과물은 List[Document] 이므로 이를 ---로 구분하는 함수
# metadata의 source를 보존하여 추가
def format_docs(docs):
    return " \n\n---\n\n ".join(['URL: '+ doc.metadata['source'] + '\nContent:' + doc.page_content for doc in docs])
    # join : 구분자를 기준으로 스트링 리스트를 하나의 스트링으로 연결


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    # retriever : question을 받아서 context 검색: document 반환
    # format_docs : document 형태를 받아서 텍스트로 변환
    # RunnablePassthrough(): 체인의 입력을 그대로 저장
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
print(format_docs(retriever.invoke("알리바바의 언어 모델 이름은?")))

URL: https://n.news.naver.com/mnews/article/020/0003684853?sid=105
Content:닫기 [기술영업人] 플리토 김재훈 팀장이 개척한 ‘언어 데이터 세일즈’의 길 입력 2025.12.26. 오전 11:55 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 ※ 21세기는 기술경쟁의 시대입니다. 수많은 빅테크 기업이 지금 이 순간도 기술과 제품을 놓고 전 세계 산업 현장에서 경쟁합니다. 그리고 이 경쟁의 현장이 바로 기술영업입니다. 기술영업은 기술적 이해가 필요한 영업으로, 주로 기업 대 기업 간 영업에서 이뤄집니다. 기술영업 전선에서는 기업의 기술력과 실력으로 경쟁하고, 그 결과가 IT 기업이 움직이는 원동력이 됩니다. 오늘날 현장에서 기술영업人들이 어떻게 경쟁하는지, 기술과 기업, 사람을 들여다보겠습니다.오늘날 AI라고 하면 챗GPT 같은 대형언어모델(LLM)을 떠올리는 사람이 많고, 실제로도 많은 AI가 LLM 모델을 기반으로 구현된다. LLM의 발전이 곧 AI의 발전이다보니 자연스레 ‘언어데이터’가 핵심 자산으로 떠올랐다. LLM 데이터의 약 절반은 언어 기반의 일반 지식 데이터고, 25%는 수학 및 추론, 25%는 코드 및 다국어 데이터 등으로 구성된다. 초창기에는 가능한 많은 매개변수와 데이터를 확보해 성능을 끌어올리는 방식이 대세였지만, 잦은 환각 현상과 데이터 비대화를 막기 위해 지금은 고품질의 정련된 언어 데이터를 활용하는 게 기본이다.이에 따라 AI 학습용 언어 데이터를 취급하는 기업, 특히 기존 번역 업계에 있던 기업들이 AI의 핵심으로 자리 잡았다. 미국의 트랜스퍼펙트는 2020년 ‘데이터포스’라는 데이터 브랜드를 출범하며 AI 언어 데이터 시장의 강자로 등극했고, 영국의 RWS나 캐나다의 텔러스 디지털도 번역에서 데이터 기업으로 변모했다. 스타트업으로는 스케일AI나 디파인드

In [ ]:
rag_chain.invoke("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?")

'**도메인 특화 언어 모델이란?**\n\n- **정의**  \n  특정 산업·분야(예: 의료, 법률, 특허, 지역 방언 등)의 전문 용어나 문체, 지식 구조를 반영하도록 사전 학습 또는 미세조정(fine‑tuning)된 대형 언어 모델(LLM)을 말합니다.  \n- **특징**  \n  - *도메인 전용 데이터* (전문 기사, 법령·특허 문서, 지역 방언 텍스트 등)를 사용해 학습 → 해당 분야에서의 정확도와 효율이 일반 LLM보다 높음  \n  - *작은 규모(4B~40억 파라미터) 모델**이라도 도메인 특화 덕분에 수백억 파라미터 대형 모델을 능가할 수 있음  \n  - 기업 내부의 AI 전환(AI\u202fTransformation, AX) 전략에서 “기술 도입”이 아니라 *가치 사슬 재편*을 위한 핵심 역량으로 활용  \n\n---\n\n**대표적인 예시**\n\n| # | 사례 | 주요 내용 |\n|---|------|-----------|\n| 1 | **Eastsoft – Alan LLM Jeju Dialect v1\u202f4B** | 제주어와 표준어 사이의 정교한 소통을 위해 40억 파라미터 규모로 미세조정. 지역 특화 AI를 통해 관광·행정 등 현장 서비스에 바로 적용 가능. |\n| 2 | **Trillium Labs – 도메인 특화 증류 기반 소형 LLM** | 의료, 법률, 과학, 행정 등 특정 영역별 ‘전문가 모델’을 개별 증류 후 MoE(Expert‑Mixing) 구조로 결합해 추론 효율과 성능을 동시에 향상. 국가 전략기술로 인정받음. |\n| 3 | **한국 특허 데이터 기반 도메인 모델** | 한국 특허청에 보유된 106개국·1억\u202f7천만 건의 출원·등록 데이터(2,500억 개 문장, 16억 장 도면)를 학습 자산으로 활용해 권리·계약·청구항 분석까지 수행. |\n| 4 | **기타** | - AI 파운데이션이 산업에 적용될 때 ‘권리 처리’와 같은 특허·IP 데이터를 포함한 도메인 모델이 필요함.<br>- 국내 기업들

In [ ]:
rag_chain.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

In [ ]:
rag_chain.invoke("알리바바가 개발한 언어 모델 이름은?")

만약 Context가 포함된 RAG 결과를 보고 싶다면, RunnableParallel을 사용하면 됩니다.

assign()을 이용하면, 체인의 결과를 받아 새로운 체인에 전달하고, 그 결과를 가져옵니다.

In [ ]:
# assign : 결과를 받아서 새로운 인수 추가하고 원래 결과와 함께 전달

from langchain_core.runnables import RunnableParallel

rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

rag_chain_with_source.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

# retriever가 1번 실행됨
# retriever의 실행 결과를 rag_chain_from_docs 에 넘겨주기 때문에

{'context': 'URL: https://n.news.naver.com/mnews/article/092/0002425168?sid=105\nContent:다른 연구소 모델을 증류하지 않고 라이선스가 없거나 불투명한 데이터에 의존하지 않는다"며 "아키텍처부터 훈련 파이프라인과 후속 훈련까지 시스템 구성요소를 직접 구축했다"고 말했다. Copyright ⓒ ZDNet Korea. All rights reserved. 무단 전재 및 재배포 금지. 이 기사는 언론사에서 IT 섹션으로 분류했습니다. 기사 섹션 분류 안내 기사의 섹션 정보는 해당 언론사의 분류를 따르고 있습니다. 언론사는 개별 기사를 2개 이상 섹션으로 중복 분류할 수 있습니다. 닫기 구독 지디넷코리아 구독하고 메인에서 바로 만나보세요!구독하고 메인에서 만나보세요! 구독중 언론사의 주요 뉴스를 메인에서 바로 만나보세요!주요 뉴스를 메인에서 만나보세요! 언론사홈 지디넷코리아 주요뉴스해당 언론사에서 선정하며 언론사(아웃링크)로 이동합니다. MS, 오픈클로 기반 AI비서 \'스카우트\' 출시…업무 자동화 확대 [SW키트] 쓴 만큼 돈 내는 \'깃허브\'…비영어권 개발자 부담 클까 MS, 빌드서 AI 에이전트 전략 공개…"개발 생태계 재편" [AI는 지금] "오픈AI만으론 부족했나"…MS, 미스트랄 품고 유럽 AI 에이전트 시장 정조준 지디넷코리아 \'홈페이지\' QR 코드를 클릭하면 크게 볼 수 있어요. QR을 촬영해보세요. 지디넷코리아 \'홈페이지\' 닫기 네이버 채널 구독하기 지디넷코리아 언론사가 직접 선정한 이슈 이슈 AI 핫트렌드 11조 실탄 쥔 中 딥시크, 조직 2배로 키운다 이슈 반도체 전쟁 반도체 미세화 한계, \'3D 적층\'으로 뚫었다…IBM, 0.7나노 시대 개막 이슈 고려아연 경영권 분쟁 영풍·MBK, \'고려아연\' 사외이사 공개 접수 마감…"경영진 이해 관계 무관" 이슈 트럼프 2.0시대 앤트로픽 고객사, 미국 정부 제소…"페이블5 차단 부당" 이전 다음 이 기사를 추천합니다 기사

In [ ]:
# Runnable Quiz

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})
# 결과 이해해보기!

이번에는 오픈 모델을 사용합니다.   
오픈 임베딩을 통해 구성한 DB와 원래 DB를 비교해 보겠습니다.

In [ ]:
uuidstr = str(uuid.uuid4())[0:6]

open_db = Chroma(embedding_function=open_embeddings, # qwen 3 emb 0.6b
                           persist_directory=f"./chroma_open_{uuidstr}", # 별도 폴더에 저장
                           collection_name='Web', # 식별 이름
                           collection_metadata={'hnsw:space':'l2'}
                           )

# 10개씩 추가
for i in tqdm(range(0, len(chunks), 10)):
    open_db.add_documents(chunks[i:min(i+10, len(chunks))])

100%|██████████| 100/100 [00:54<00:00,  1.84it/s]


In [ ]:
open_retriever = open_db.as_retriever(search_kwargs={'k':5})

In [ ]:
open_retriever.invoke("알리바바의 언어 모델은 무엇입니까?")

In [ ]:
rag_chain_open = (
    {"context": open_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source_open = RunnableParallel(
    {"context": open_retriever | format_docs, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

In [ ]:
questions = ["도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?",
             "인공지능의 최근 발전 방식은? 관련 링크도 보여주세요",
             "알리바바의 언어 모델 이름은?"]

In [ ]:
# Retriever 비교

result_open = open_retriever.batch(questions)

for i in range(3):
    print(f'Question: {questions[i]}')
    for j in result_open[i]:
        print('   ',j.page_content[0:50])

Question: 도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?
    도메인 특화 모델 개발은 범용 모델 활용보다 훨씬 까다롭다. 온톨로지 설계와 정밀한 데이터
    도메인 지식은 100% 자사 서버에 축적하여 데이터 주권을 확보해야 합니다. 외국 모델을 
    전면에 나섰다"며 "단일 모델 의존 자체가 핵심 리스크"라고 진단했다. 이어 "업무 유형과
    닫기 딥서치·모티프테크놀로지스, 금융 특화 AI 모델 공동 개발 추진 입력 2026.04.
    닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 입력 2026.
Question: 인공지능의 최근 발전 방식은? 관련 링크도 보여주세요
    투데이] 와이즈넛, 고용정보원 AI 서비스 개발 外 [ZD SW 투데이] 트웰브랩스, '아
    열린 '딜로이트 커넥트 코리아 2026' 행사장에서 진행된 그룹 인터뷰에서 기자들의 질문에
    대한 기본적인 학습을 먼저 수행합니다. 그다음에 이 환자 데이터를 가지고 추가적으로 학습을
    주요 뉴스를 메인에서 바로 만나보세요!주요 뉴스를 메인에서 만나보세요! 언론사홈 네이버에서
    메인에서 만나보세요! 구독중 언론사의 주요 뉴스를 메인에서 바로 만나보세요!주요 뉴스를 메
Question: 알리바바의 언어 모델 이름은?
    닫기 이스트소프트, 제주 특화 AI 언어모델 오픈소스 공개 입력 2026.02.09. 오후
    닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 입력 2026.
    이어 AI 바이오프린팅 기반 장기 재생 임상에도 진입한다는 계획이다.◆ 셀바스AI, 한·중
    언어 모델)을 만들려 한다. 이런 분야에도 협력해 지원 분야를 넓히고 강점을 가져가고 싶다
    닫기 LGU+, 세계 최고 권위 AI 학회에 익시젠 논문 채택…“산업 특화 기술력 입증” 


In [ ]:
result_openai = retriever.batch(questions)
result_openai
for i in range(3):
    print(f'Question: {questions[i]}')
    for j in result_openai[i]:
        print('   ',j.page_content[0:50])

Question: 도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?
    도메인 지식은 100% 자사 서버에 축적하여 데이터 주권을 확보해야 합니다. 외국 모델을 
    닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 입력 2026.
    닫기 AI 모델 최적화도 국가전략기술 인정 입력 2025.12.18. 오후 12:01 기사
    닫기 이스트소프트, 제주 특화 AI 언어모델 오픈소스 공개 입력 2026.02.09. 오후
    것은 단순한 텍스트가 아니라 AI 파운데이션이 산업으로 내려갈 때 부딪히는 권리 처리의 자
Question: 인공지능의 최근 발전 방식은? 관련 링크도 보여주세요
    다른 연구소 모델을 증류하지 않고 라이선스가 없거나 불투명한 데이터에 의존하지 않는다"며 
    오픈AI는 현재 AI 모델 업계에서 가장 주목받는 기업들이다. 양사는 각각 ‘챗GPT’와 
    Need)’의 공동 저자이기도 하다.이 논문은 대규모 언어 모델(LLM)의 핵심 기술인 트
    이용해 거대한 네트워크를 학습시켜 구글의 정보 구성 방식을 근본적으로 바꾸자’고 제안했지만
    개발자 생태계 확대에 속도를 내고 있다.오픈AI도 국내 기업·기관 시장에서 접점을 빠르게 
Question: 알리바바의 언어 모델 이름은?
    닫기 [기술영업人] 플리토 김재훈 팀장이 개척한 ‘언어 데이터 세일즈’의 길 입력 2025
    닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 입력 2026.
    닫기 이스트소프트, 제주 특화 AI 언어모델 오픈소스 공개 입력 2026.02.09. 오후
    닫기 대형 LLM 중심 구조 흔들…소형 모델·엣지 AI 부상[AI 7대 트렌드④] 입력 2
    닫기 MS, 멀티모달 AI 7종 공개…"자체 모델로 경쟁" 입력 2026.06.03. 오전


In [ ]:
# 최종 결과 비교
print(rag_chain_open.batch(questions))

ConnectError: [Errno 111] Connection refused

In [ ]:
print(rag_chain.batch(questions))

ConnectError: [Errno 111] Connection refused

# RAG 성능 평가하기   

구성한 RAG의 성능은 어떻게 평가할까요?   

RAGAS (https://docs.ragas.io/en/stable/)는 다양한 메트릭을 통한 RAG의 성능 평가를 지원합니다.

RAG의 평가를 위해서는 정답이 있는 Q/A 데이터가 필요합니다.   
실습 시트에서 eval.jsonl을 다운로드하여 불러옵니다.

In [ ]:
import pandas as pd
df = pd.read_csv('./eval.csv')
eval_dataset = df.to_dict('list')
eval_dataset

In [ ]:
questions, ground_truths = eval_dataset['questions'], eval_dataset['ground_truths']

In [ ]:
for i in range(len(questions)):
    print(f'#{i}')
    print(f'Question: {questions[i]}\n')
    print(f'Ground Truth: {ground_truths[i]}\n')
    print('-----------')

구성된 RAG 체인을 이용해, RAGAS의 Evaluate에 필요한 데이터를 구성합니다.

In [ ]:
dataset = []

for query,reference in zip(questions,ground_truths):

    relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
    response = rag_chain.invoke(query)
    dataset.append(
        {
            "user_input":query,
            "retrieved_contexts":relevant_docs,
            "response":response,
            "reference":reference
        }
        # 질문, 검색결과, 답변, 정답(레퍼런스)
    )

In [ ]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset)
evaluation_dataset

RAGAS는 LLM을 이용해 정답과 답변을 개별 Claim(주장)으로 분할합니다.

이후, `LLMContextRecall`, `Faitufulness`, `FactualCorrectness` 등의 다양한 메트릭을 통해 RAG 파이프라인의 성능을 평가합니다.   
LLM 기반의 방법이므로 평가 LLM의 선정이 중요하며, 절대 수치보다는 상대적 비교가 효과적입니다.


- Context Recall: 정답의 Claim들이 모두 검색됐는가
- Faithfulness : 답변의 Claim이 얼마나 검색 결과에 근거했는가
- Factual Correctness : 정답과 답변의 Claim이 얼마나 일치하는가
- Bleu Score : 정답과 답변 키워드가 얼마나 일치하는가
- Semantic Sim : 정답 답변 임베딩이 얼마나 가까운가   


In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from ragas.metrics import BleuScore, SemanticSimilarity
from langchain_openai import ChatOpenAI

# https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

# 평가자 LLM
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))

# 평가자 Embedding
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model = 'text-embedding-3-large'))
semantic_scorer = SemanticSimilarity(embeddings = evaluator_embeddings)


result = evaluate(dataset=evaluation_dataset,
                  metrics=[BleuScore(), LLMContextRecall(), semantic_scorer, Faithfulness(), FactualCorrectness()],
                  llm=evaluator_llm,
                  embeddings = evaluator_embeddings
                  )
result

In [ ]:
result.scores

In [ ]:
result.to_pandas()

In [ ]:
# dict로 변경하기(복잡..)
import ast

result_str = str(result)
result_dict = ast.literal_eval(result_str)
result_dict

# [실습] 오픈 임베딩 체인의 성능 평가하기   

다음의 코드를 이용해, 오픈 임베딩을 비롯한 다양한 파이프라인의 모델 성능도 평가해 보세요.   

새로운 Ollama 모델을 쓰고 싶은 경우, `ollama pull`을 통해 저장하고   
`ollama stop`을 통해 기존 모델을 해제한 뒤 실행해 주세요.

In [ ]:
MODEL_NAME='gemma3:12b'
# 'gpt-'를 통해 gpt-oss 이외의 OpenAI 모델 불러올 수 있음

TEMPERATURE=0
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 10
embedding_method = 'QWEN' # or 'QWEN'


# CSV 파일 경로 설정
csv_path = "experiments/experiment_log.csv"
os.makedirs("experiments", exist_ok=True)


def RAG_pipeline():

    emb_map = {'QWEN':open_embeddings, 'OPENAI':openai_embeddings}

    dataset = []

    if 'gpt' in MODEL_NAME:
        llm = ChatOpenAI(model=MODEL_NAME, temperature = TEMPERATURE)
    else:
        try: # Reasoning Model에 한해 Path 분리
            llm = ChatOllama(model=MODEL_NAME, temperature = TEMPERATURE, reasoning = True)
            llm.invoke("hello!")
        except:
            llm = ChatOllama(model=MODEL_NAME, temperature = TEMPERATURE)

    print('## LLM 세팅 완료')
    Chroma().delete_collection() # 메모리 기존 데이터 삭제

    preprocessed_docs = load_docs_from_jsonl("eval.jsonl")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    print('## DB 구성 중...')

    chunks = text_splitter.split_documents(preprocessed_docs)
    print(f'## Chunks 생성 완료... 총 {len(chunks)} 개 청크')
    db = Chroma(embedding_function=emb_map[embedding_method],
                collection_metadata={'hnsw:space':'l2'},
                )

    for i in tqdm(range(0, len(chunks), 10)):
        db.add_documents(chunks[i:min(i+10, len(chunks))])

    print('## DB 구성 완료...')

    retriever = db.as_retriever(search_kwargs={'k':TOP_K})

    rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser())

    for query,reference in tqdm(zip(questions,ground_truths)):

        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        response = rag_chain.invoke(query)
        dataset.append(
            {
                "user_input":query,
                "retrieved_contexts":relevant_docs,
                "response":response,
                "reference":reference
            }
        )

    print('## 답변 생성 완료...')
    evaluation_dataset = EvaluationDataset.from_list(dataset)

    result = evaluate(dataset=evaluation_dataset,
                    metrics=[BleuScore(), LLMContextRecall(), semantic_scorer, Faithfulness(), FactualCorrectness()],
                    llm=evaluator_llm,
                    embeddings = evaluator_embeddings
                    )

    result_str = str(result)
    result_dict = ast.literal_eval(result_str)
    return result_dict


실험 결과를 저장할 수 있습니다.

In [ ]:
import csv
import datetime
import os

result = RAG_pipeline()

timestamp = datetime.datetime.now().isoformat()
row = {
    "timestamp": timestamp,
    "model_name": MODEL_NAME,
    "embedding_method": embedding_method,
    "temperature": TEMPERATURE,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    **result
}

# 파일이 없으면 헤더 추가
write_header = not os.path.exists(csv_path)

with open(csv_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(row)

print(f"[{timestamp}] {MODEL_NAME} {embedding_method}| temp={TEMPERATURE}, chunk={CHUNK_SIZE}/{CHUNK_OVERLAP}, top_k={TOP_K} | results={result}")
